# Bimodal Gaussian ST + Flow Experiment

CRAFT-style Simulated Tempering on a 2-mode mixture:

$$\pi = \tfrac13\,\mathcal N(\mu_L, 5I) + \tfrac23\,\mathcal N(\mu_R, 5I),\quad \mu_L=(-5,0),\ \mu_R=(5,0)$$

Easy reference $\pi_0 = \mathcal N(0, I)$. Bridge: $\log\gamma_i = (1-\beta_i)\log\pi_0 + \beta_i\log\pi$ with $\beta_i = i/(T-1)$.

Outer loop: **implement → tune_weight → learn_flow**.

If imports fail, restart the kernel with the project venv and ensure `src` is on `PYTHONPATH` (cell below).

In [1]:
from pathlib import Path
import sys

import jax.numpy as jnp
from jax import random
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from krystal.tempering import Pipeline
from krystal.transport import NICE

## Target / bridge helpers

In [2]:
DIM = 2
VAR = 5.0
MEAN_LEFT = jnp.array([-5.0, 0.0])
MEAN_RIGHT = jnp.array([5.0, 0.0])
W_LEFT = 1.0 / 3.0
W_RIGHT = 2.0 / 3.0


def _gauss_logpdf(x, mean, var: float):
    dim = x.shape[-1]
    return -0.5 * (
        dim * jnp.log(2.0 * jnp.pi * var) + jnp.sum((x - mean) ** 2, axis=-1) / var
    )


def target_log_prob(x):
    a = jnp.log(W_LEFT) + _gauss_logpdf(x, MEAN_LEFT, VAR)
    b = jnp.log(W_RIGHT) + _gauss_logpdf(x, MEAN_RIGHT, VAR)
    return jnp.logaddexp(a, b)


def easy_log_prob(x):
    return _gauss_logpdf(x, jnp.zeros(DIM), 1.0)


def make_log_prob(n_levels: int):
    def log_prob(level: int, x):
        beta = level / max(n_levels - 1, 1)
        return (1.0 - beta) * easy_log_prob(x) + beta * target_log_prob(x)

    return log_prob


def mode_weights(samples):
    d_left = jnp.sum((samples - MEAN_LEFT) ** 2, axis=-1)
    d_right = jnp.sum((samples - MEAN_RIGHT) ** 2, axis=-1)
    left = d_left < d_right
    frac_left = float(jnp.sum(left) / samples.shape[0])
    return frac_left, 1.0 - frac_left


print(f"target: {W_LEFT:.3f} N({MEAN_LEFT.tolist()}, {VAR}I) + {W_RIGHT:.3f} N({MEAN_RIGHT.tolist()}, {VAR}I)")

target: 0.333 N([-5.0, 0.0], 5.0I) + 0.667 N([5.0, 0.0], 5.0I)


## Hyperparameters

Larger particle count + a dedicated **collect** phase after training to accumulate many target-temperature states.

(Slower — ST loop is pure Python.)

In [3]:
n_levels = 5
n_particles = 256
n_outer = 25
n_inner = 60
# After training: run ST-only sweeps and keep every visit to the target level.
n_collect_rounds = 40
collect_inner = 80

pipe = Pipeline(
    n_levels=n_levels,
    flow=NICE(dim=DIM, n_layers=4, hidden_dims=(32, 32)),
    log_prob=make_log_prob(n_levels),
    n_inner=n_inner,
    n_local=3,
    local_step_size=1.0,
    lr=1e-3,
    learn_steps=5,
    eta=0.5,
)

key = random.PRNGKey(0)
key, k0, k1 = random.split(key, 3)
x0 = random.normal(k0, (n_particles, DIM))
state = pipe.init_state(k1, x0)

history = {"occupancy": [], "log_lambda": []}
print(
    f"train: outer={n_outer}, inner={n_inner}, particles={n_particles}, levels={n_levels}\n"
    f"collect: rounds={n_collect_rounds}, inner={collect_inner}"
)

train: outer=25, inner=60, particles=256, levels=5
collect: rounds=40, inner=80


## Train (implement → tune_weight → learn_flow)

In [ ]:
for t in tqdm(range(n_outer), desc="train (outer)"):
    state = pipe.step(state, progress=True)
    occ = state.counts / jnp.maximum(state.counts.sum(), 1)
    history["occupancy"].append(occ)
    history["log_lambda"].append(state.log_lambda)
    tqdm.write(
        f"[{t+1:02d}/{n_outer}] occupancy={occ}  log_lambda={state.log_lambda}"
    )

train (outer):   0%|          | 0/25 [00:00<?, ?it/s]

implement (ST):   0%|          | 0/60 [00:00<?, ?it/s]

learn_flow:   0%|          | 0/4 [00:00<?, ?it/s]

[01/25] occupancy=[0.44850263 0.21341147 0.11223959 0.07076823 0.15507813]  log_lambda=[-0.5036459  -0.13229978  0.18899322  0.41960585  0.02734667]


implement (ST):   0%|          | 0/60 [00:00<?, ?it/s]

learn_flow:   0%|          | 0/4 [00:00<?, ?it/s]

... (7 outputs left out)

implement (ST):   0%|          | 0/60 [00:00<?, ?it/s]

learn_flow:   0%|          | 0/4 [00:00<?, ?it/s]

[05/25] occupancy=[0.20260417 0.201237   0.19557293 0.1902995  0.21028647]  log_lambda=[-0.61166054  0.03264248  0.5432847   0.6767228  -0.6409896 ]


implement (ST):   0%|          | 0/60 [00:00<?, ?it/s]

## Collect many target-temperature states

Freeze flows/λ and keep running `implement`, appending every particle that sits at the target level after each round.

In [ ]:
target_level = n_levels - 1
pipe.n_inner = collect_inner

collected = []
for r in tqdm(range(n_collect_rounds), desc="collect (target)"):
    state = pipe.implement(state, progress=True)
    batch = state.x[state.levels == target_level]
    if batch.shape[0] > 0:
        collected.append(batch)
    tqdm.write(
        f"collect [{r+1:02d}/{n_collect_rounds}] at target this round: {batch.shape[0]}"
    )

if not collected:
    at_target = jnp.zeros((0, DIM))
    print("No particles at target — ladder did not mix enough.")
else:
    at_target = jnp.concatenate(collected, axis=0)
    frac_left, frac_right = mode_weights(at_target)
    emp_mean = jnp.mean(at_target, axis=0)
    true_mean = W_LEFT * MEAN_LEFT + W_RIGHT * MEAN_RIGHT
    print(f"\ntotal target samples: {at_target.shape[0]}")
    print(f"empirical mode weights: left={frac_left:.3f} right={frac_right:.3f}")
    print(f"true mode weights:      left={W_LEFT:.3f} right={W_RIGHT:.3f}")
    print(f"empirical mean: {emp_mean}")
    print(f"true mean:      {true_mean}")

    weight_ok = abs(frac_right - W_RIGHT) < 0.15 and abs(frac_left - W_LEFT) < 0.15
    mean_ok = float(jnp.linalg.norm(emp_mean - true_mean)) < 1.5
    print(f"weight fit (tol 0.15): {'OK' if weight_ok else 'WEAK'}")
    print(f"mean fit (L2 < 1.5):   {'OK' if mean_ok else 'WEAK'}")

## Plots: occupancy + ST target samples vs target PDF

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# 1) Occupancy over outer iterations
occ = jnp.stack(history["occupancy"])
for i in range(n_levels):
    axes[0].plot(occ[:, i], label=f"level {i}")
axes[0].axhline(1 / n_levels, color="k", ls="--", lw=0.8, label="uniform")
axes[0].set_xlabel("outer iter")
axes[0].set_ylabel("occupancy")
axes[0].set_title("Ladder occupancy")
axes[0].legend(fontsize=8)

# Target PDF on a grid
x_min, x_max = -12.0, 12.0
y_min, y_max = -8.0, 8.0
gs = 200
xx, yy = jnp.meshgrid(
    jnp.linspace(x_min, x_max, gs),
    jnp.linspace(y_min, y_max, gs),
)
grid = jnp.stack([xx.ravel(), yy.ravel()], axis=-1)
pdf = jnp.exp(target_log_prob(grid)).reshape(xx.shape)

# 2) Target PDF contours + all collected ST particles
axes[1].contourf(xx, yy, pdf, levels=30, cmap="Blues")
axes[1].contour(xx, yy, pdf, levels=8, colors="k", linewidths=0.4, alpha=0.5)
if at_target.shape[0] > 0:
    axes[1].scatter(
        at_target[:, 0],
        at_target[:, 1],
        s=8,
        c="C3",
        alpha=0.35,
        label=f"ST target (n={at_target.shape[0]})",
    )
axes[1].scatter(*MEAN_LEFT, c="C1", marker="x", s=100, zorder=5, label="mode L")
axes[1].scatter(*MEAN_RIGHT, c="C2", marker="x", s=100, zorder=5, label="mode R")
axes[1].set_xlim(x_min, x_max)
axes[1].set_ylim(y_min, y_max)
axes[1].set_xlabel("x0")
axes[1].set_ylabel("x1")
axes[1].set_title("ST particles vs target PDF")
axes[1].legend(fontsize=8, loc="upper left")
axes[1].set_aspect("equal", adjustable="box")

# 3) 1D marginal on x0: histogram vs target marginal PDF
# For isotropic mixture, marginal of x0 is (1/3)N(-5,5)+(2/3)N(5,5).
x_line = jnp.linspace(x_min, x_max, 400)


def _gauss1d(x, mean, var):
    return jnp.exp(-0.5 * ((x - mean) ** 2) / var) / jnp.sqrt(2.0 * jnp.pi * var)


marginal = W_LEFT * _gauss1d(x_line, -5.0, VAR) + W_RIGHT * _gauss1d(x_line, 5.0, VAR)
if at_target.shape[0] > 0:
    axes[2].hist(
        at_target[:, 0],
        bins=40,
        density=True,
        color="C3",
        alpha=0.45,
        label="ST hist (x0)",
    )
axes[2].plot(x_line, marginal, "k-", lw=2, label="target marginal")
axes[2].set_xlabel("x0")
axes[2].set_ylabel("density")
axes[2].set_title("x0 marginal: samples vs PDF")
axes[2].legend(fontsize=8)

fig.tight_layout()
plt.show()